# Fundamentos de Aprendizagem Profunda

## Estudo 1 — CIFAR-10: hiperparâmetros de uma ANN


Este estudo usa uma ANN para classificar imagens CIFAR-10 em tons de cinzento. A execução atual treina três configurações de taxa de aprendizagem e guarda as métricas em CSV. As secções históricas mostram ensaios recuperados do trabalho original; os seus resultados devem ser interpretados com as limitações indicadas em Resultados.


## Bibliotecas e configuração


In [ ]:
# Imports

# PyTorch e PyTorch Lightning
import torch
from torch import nn
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from torch.utils.data import DataLoader, random_split
import torch.nn.init as init
from torchmetrics.classification import Precision, Recall

from torchvision import datasets, transforms

import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt



# Resolve paths from either the project root or the notebooks directory.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "notebooks").is_dir():
    raise FileNotFoundError("Open this project or its notebooks directory before running.")
DATA_DIR = PROJECT_ROOT / "data" / "cifar10"
RESULTS_DIR = PROJECT_ROOT / "results" / "cifar10"


## Dados

In [ ]:
# Transformações aplicadas às imagens para preparação do dataset, o objetivo é simplificar e normalizar a informação antes do treino.
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  # Converte para escala de cinza
    transforms.ToTensor(),                        # Converte para tensor
    transforms.Normalize((0.5,), (0.5,))         # Normaliza para média=0.5 e desvio=0.5
])


# Download e preparação do dataset CIFAR-10 com as transformações definidas acima.
train_dataset = datasets.CIFAR10(
    root=str(DATA_DIR),          # Diretório onde os dados serão armazenados
    train=True,             # Indica que é o dataset de treino
    download=True,          # Faz download automaticamente se os dados não estiverem presentes
    transform=transform     # Aplica as transformações definidas anteriormente (grayscale + tensor)
)

test_dataset = datasets.CIFAR10(
    root=str(DATA_DIR),          # Mesmo diretório para o dataset de teste
    train=False,            # Indica que é o dataset de teste
    download=True,          # Download automático se necessário
    transform=transform     # Aplica as mesmas transformações do treino
)

# Keep the original CIFAR-10 test set separate from training and validation.
val_size = int(0.1 * len(train_dataset))
train_size = len(train_dataset) - val_size
train_subset, val_subset = random_split(
    train_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


## Modelos


In [ ]:
# Modelo ( 
            # input_size = Tamanho da entrada (32x32px),
            # hidden_units = Lista de neurônios por camada oculta,
            # num_classes = Número de classes de saída (CIFAR-10)
            # lr = Learning rate para o optimizer
            # af = Função de ativação
            # weight_init = Pesos iniciais
            # optimizer_class = Classe do optimizer
        #)


class SimpleANN(pl.LightningModule):
    def __init__(self, input_size=32*32, hidden_units=[256, 256], num_classes=10, lr=0.0001, af=nn.ReLU, optimizer_class=torch.optim.Adam, weight_init="default", dropout_rate=0.2 ):
        super().__init__()


        self.metrics_log = []           # Lista para armazenar métricas de train e val
        self.lr = lr                    # Learning rate
        self.af = af                    # Função de ativação
        self.weight_init = weight_init  # Pesos iniciais
        self.optimizer_class = optimizer_class  # Classe do optimizer
        self.dropout_rate = dropout_rate        # Drop out

        self.flatten = nn.Flatten()     # Flatten: transforma a imagem 2D em um vetor 1D. acreditamos que istp é necessário porque estamos a usar uma ANN totalmente conectada (fully connected) e não uma CNN, 
                                        # ou seja, não precisamos manter a informação espacial das imagens.
 

        # Construtor das camadas do modelo de forma dinâmica. Esta adaptação permite tornar o processo mais rápido, bastando definir as camadas e seus neurônios, por exemplo: hidden_units=[256, 256]
        layers = []
        in_features = input_size

        # Hidden layers
        for h in hidden_units:                          # Loop que compõe as camadas de acordo com hidden_units=[256, 256]
            layers.append(nn.Linear(in_features, h))    # Camada linear (fully connected)
            layers.append(af())                         # Função de ativação
            if self.dropout_rate > 0:                   # aplica dropout apenas se taxa > 0
                layers.append(nn.Dropout(p=self.dropout_rate))
            in_features = h                             # Atualiza o número de entradas para a próxima camada
            
        layers.append(nn.Linear(in_features, num_classes))  # Adiciona a camada final com o número de saídas igual ao número de classes

        self.model = nn.Sequential(*layers)                 # Cria o modelo sequencial e define a variável de acesso self.model

        self._init_weights()  # Pesos

        # métricas
        self.train_precision = Precision(task="multiclass", num_classes=num_classes, average='macro')
        self.train_recall = Recall(task="multiclass", num_classes=num_classes, average='macro')
        self.val_precision = Precision(task="multiclass", num_classes=num_classes, average='macro')
        self.val_recall = Recall(task="multiclass", num_classes=num_classes, average='macro')


    # Pesos de inicio
    def _init_weights(self):
        for m in self.model:
            if isinstance(m, nn.Linear):
                if self.weight_init == "default":
                    continue                        # Usa inicialização padrão do PyTorch
                elif self.weight_init == "xavier":  # Xavier
                    init.xavier_uniform_(m.weight)
                elif self.weight_init == "he":      # HE
                    init.kaiming_uniform_(m.weight, nonlinearity='relu')
                elif self.weight_init == "normal":  # Normal
                    init.normal_(m.weight, mean=0.0, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)


    def forward(self, x):
        x = self.flatten(x)     # Flatten: converte a imagem 2D em vetor 1D para entrada em uma ANN totalmente conectada.
        return self.model(x)    # Passa o vetor pelo modelo sequencial de camadas

    # Executa treino calcula logits, loss (CrossEntropy) e acc, e registra métricas.
    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = nn.CrossEntropyLoss()(logits, y)
        
        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()
        
        # Calcula precision e recall usando torchmetrics
        precision = self.train_precision(preds, y)
        recall = self.train_recall(preds, y)
        
        # Log das métricas
        self.log('train_loss', loss, prog_bar=True)
        self.log('train_acc', acc, prog_bar=True)
        self.log('train_precision', precision, prog_bar=True)
        self.log('train_recall', recall, prog_bar=True)
        
        return loss


    # Executa  validação para um batch.
    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = nn.CrossEntropyLoss()(logits, y)
        
        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()
        
        # Calcula precision e recall usando torchmetrics
        precision = self.val_precision(preds, y)
        recall = self.val_recall(preds, y)
        
        # Log das métricas
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_acc', acc, prog_bar=True)
        self.log('val_precision', precision, prog_bar=True)
        self.log('val_recall', recall, prog_bar=True)
        
        return loss

    # Define o otimizador que atualiza os pesos do modelo durante o treino, lr e optimizer_class é drivado de arg
    def configure_optimizers(self):
        return self.optimizer_class(self.parameters(), lr=self.lr)
    
    # Ao final de cada epoch de validação, coleta e armazena métricas de treino e validação em um log, permite análise posterior das curvas de loss e acc.
    def on_validation_epoch_end(self):
        metrics = {
            "epoch": int(self.current_epoch),
            "train_loss": self._to_float(self.trainer.callback_metrics.get("train_loss")),
            "train_acc": self._to_float(self.trainer.callback_metrics.get("train_acc")),
            "train_precision": self._to_float(self.trainer.callback_metrics.get("train_precision")),
            "train_recall": self._to_float(self.trainer.callback_metrics.get("train_recall")),
            "val_loss": self._to_float(self.trainer.callback_metrics.get("val_loss")),
            "val_acc": self._to_float(self.trainer.callback_metrics.get("val_acc")),
            "val_precision": self._to_float(self.trainer.callback_metrics.get("val_precision")),
            "val_recall": self._to_float(self.trainer.callback_metrics.get("val_recall")),
        }
        self.metrics_log.append(metrics)

    # Apenas converte tensor(0.1000) > 0.1000, facilitar armazenamento e manipulação
    @staticmethod
    def _to_float(x):
        if x is None:
            return None
        if hasattr(x, "item"):
            return float(x.item())
        return float(x)

## Treino e avaliação

In [ ]:

# {
#     "lr": taxa de aprendizagem (float),
#     "hidden_units": lista com o número de neurônios por camada oculta (list[int]),
#     "af": função de ativação (ex: nn.ReLU, nn.Tanh, nn.Sigmoid),
#     "optimizer_class": classe do otimizador (ex: torch.optim.Adam, torch.optim.SGD),
#     "weight_init": método de inicialização dos pesos ("default", "xavier", "he", "normal"),
#     "dropout_rate": taxa de dropout (float entre 0.0 e 1.0)
# }


# Se o prof quiser testar, pode receber varias, cada um dentro {},{}
# {"lr": 0.001, "hidden_units": [512, 256, 128, 64, 32], "af": nn.ReLU, "optimizer_class": torch.optim.Adam, "weight_init": "normal", "dropout_rate": 0.2}
# {"lr": 0.0005, "hidden_units": [64, 32, 32], "af": nn.Tanh, "optimizer_class": torch.optim.SGD, "weight_init": "xavier", "dropout_rate": 0.1}
# {"lr": 0.001, "hidden_units": [128, 64], "af": nn.Sigmoid, "optimizer_class": torch.optim.RMSprop, "weight_init": "he", "dropout_rate": 0.0}

hyperparams = [
    {"lr": 0.001, "hidden_units": [512, 256, 128, 64, 32], "af": nn.ReLU, "optimizer_class": torch.optim.Adam,"weight_init": "he", "dropout_rate": 0.2},
    {"lr": 0.1, "hidden_units": [512, 256, 128, 64, 32], "af": nn.ReLU, "optimizer_class": torch.optim.Adam,"weight_init": "he", "dropout_rate": 0.2},
    {"lr": 0.01, "hidden_units": [512, 256, 128, 64, 32], "af": nn.ReLU, "optimizer_class": torch.optim.Adam,"weight_init": "he", "dropout_rate": 0.2},
    ]


RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Armazenamento temporário dos resultados
results = []


# Loop pelos diferentes hiperparâmetros definidos em `hyperparams`
# i: índice do teste, hp: dicionário com os hiperparâmetros para os modelos
for i, hp in enumerate(hyperparams, start=1):

    print(f"\n  Model {i}")
    
    # Instancia o modelo com os hiperparâmetros atuais
    model = SimpleANN(
        input_size=32 * 32,
        hidden_units=hp["hidden_units"],
        num_classes=10,
        lr=hp["lr"],
        af=hp["af"],
        optimizer_class=hp["optimizer_class"],
        weight_init=hp["weight_init"],
        dropout_rate=hp["dropout_rate"]
    )

    # Configura o trainer do PyTorch Lightning  
    trainer = Trainer(
        max_epochs=32,
        accelerator="gpu",
        devices=1,
        enable_progress_bar=True,
        enable_checkpointing=False,
        logger=False
    )

    # Treina o modelo usando os loaders de treino e validação
    trainer.fit(model, train_loader, val_loader)

    # Criar DataFrame com as métricas
    # Salva em CSV dentro do loop. Por padrão, o arquivo terá o nome `resultsModel_defaultX.csv`
    # !!! Para guardar sem sobrescrever, é preciso alterar manualmente o nome do CSV. !!!
    df = pd.DataFrame(model.metrics_log)
    df = df.dropna() 
    df.to_csv(RESULTS_DIR / f"resultsModel_default{i}.csv", index=False)
    print(df)



## Resultados


O primeiro gráfico mostra os três treinos atuais. Os restantes gráficos mostram CSV históricos recuperados, agrupados por tema. Nos ensaios antigos, a coluna `val` foi obtida usando o conjunto de teste como validação; não compares os seus valores diretamente com os da execução atual.


In [ ]:
def plot_experiment_group(prefix, title, historical=False):
    files = sorted(RESULTS_DIR.glob(f"resultsModel_{prefix}*.csv"))
    if not files:
        print(f"Sem CSV para {title.lower()}.")
        return

    datasets = [(file.stem, pd.read_csv(file)) for file in files]
    metrics = [
        ("loss", "Perda"), ("acc", "Exatidão"),
        ("precision", "Precisão"), ("recall", "Sensibilidade"),
    ]
    metrics = [
        (key, label) for key, label in metrics
        if all(f"train_{key}" in df and f"val_{key}" in df for _, df in datasets)
    ]
    fig, axes = plt.subplots(1, len(metrics), figsize=(5 * len(metrics), 4), squeeze=False)
    for ax, (key, label) in zip(axes[0], metrics):
        for name, df in datasets:
            ax.plot(df["epoch"], df[f"train_{key}"], label=f"{name} · treino")
            ax.plot(df["epoch"], df[f"val_{key}"], label=f"{name} · val.")
        ax.set(title=label, xlabel="Época", ylabel=label)
        ax.grid(True)
        ax.legend(fontsize=7)
    fig.suptitle(title + (" · CSV históricos" if historical else " · execução atual"))
    fig.tight_layout()
    plt.show()

plot_experiment_group("default", "Três treinos atuais")


### Modelo de referência

#### Registos históricos

Os valores abaixo são a última linha de cada CSV original. A coluna `val` foi calculada com o conjunto de teste reutilizado como validação; não representa um teste independente. Os CSV não guardam a configuração completa dos modelos.

| CSV | Épocas registadas | `val_loss` final | `val_acc` final |
| --- | ---: | ---: | ---: |
| `resultsModel_base1.csv` | 32 | 1.575 | 0.462 |


In [ ]:
plot_experiment_group("base", "Modelo de referência", historical=True)


### Taxa de aprendizagem

#### Registos históricos

Os valores abaixo são a última linha de cada CSV original. A coluna `val` foi calculada com o conjunto de teste reutilizado como validação; não representa um teste independente. Os CSV não guardam a configuração completa dos modelos.

| CSV | Épocas registadas | `val_loss` final | `val_acc` final |
| --- | ---: | ---: | ---: |
| `resultsModel_lr1.csv` | 16 | 2.107 | 0.453 |
| `resultsModel_lr2.csv` | 16 | 2.303 | 0.100 |
| `resultsModel_lr3.csv` | 16 | 2.306 | 0.100 |


In [ ]:
plot_experiment_group("lr", "Taxa de aprendizagem", historical=True)


### Número de camadas e unidades

#### Registos históricos

Os valores abaixo são a última linha de cada CSV original. A coluna `val` foi calculada com o conjunto de teste reutilizado como validação; não representa um teste independente. Os CSV não guardam a configuração completa dos modelos.

| CSV | Épocas registadas | `val_loss` final | `val_acc` final |
| --- | ---: | ---: | ---: |
| `resultsModel_nlu1.csv` | 16 | 1.751 | 0.431 |
| `resultsModel_nlu2.csv` | 16 | 2.299 | 0.438 |
| `resultsModel_nlu3.csv` | 16 | 2.429 | 0.446 |


In [ ]:
plot_experiment_group("nlu", "Número de camadas e unidades", historical=True)


### Funções de ativação

#### Registos históricos

Os valores abaixo são a última linha de cada CSV original. A coluna `val` foi calculada com o conjunto de teste reutilizado como validação; não representa um teste independente. Os CSV não guardam a configuração completa dos modelos.

| CSV | Épocas registadas | `val_loss` final | `val_acc` final |
| --- | ---: | ---: | ---: |
| `resultsModel_af1.csv` | 16 | 2.029 | 0.451 |
| `resultsModel_af2.csv` | 16 | 1.900 | 0.389 |
| `resultsModel_af3.csv` | 16 | 1.916 | 0.399 |


In [ ]:
plot_experiment_group("af", "Funções de ativação", historical=True)


### Otimizadores

#### Registos históricos

Os valores abaixo são a última linha de cada CSV original. A coluna `val` foi calculada com o conjunto de teste reutilizado como validação; não representa um teste independente. Os CSV não guardam a configuração completa dos modelos.

| CSV | Épocas registadas | `val_loss` final | `val_acc` final |
| --- | ---: | ---: | ---: |
| `resultsModel_op1.csv` | 16 | 2.011 | 0.456 |
| `resultsModel_op2.csv` | 16 | 2.303 | 0.100 |
| `resultsModel_op3.csv` | 16 | 2.097 | 0.391 |


In [ ]:
plot_experiment_group("op", "Otimizadores", historical=True)


### Tamanho do batch

#### Registos históricos

Os valores abaixo são a última linha de cada CSV original. A coluna `val` foi calculada com o conjunto de teste reutilizado como validação; não representa um teste independente. Os CSV não guardam a configuração completa dos modelos.

| CSV | Épocas registadas | `val_loss` final | `val_acc` final |
| --- | ---: | ---: | ---: |
| `resultsModel_bs1.csv` | 16 | 1.828 | 0.352 |
| `resultsModel_bs2.csv` | 16 | 1.846 | 0.340 |
| `resultsModel_bs3.csv` | 16 | 1.802 | 0.354 |


In [ ]:
plot_experiment_group("bs", "Tamanho do batch", historical=True)


### Número de épocas

#### Registos históricos

Os valores abaixo são a última linha de cada CSV original. A coluna `val` foi calculada com o conjunto de teste reutilizado como validação; não representa um teste independente. Os CSV não guardam a configuração completa dos modelos.

| CSV | Épocas registadas | `val_loss` final | `val_acc` final |
| --- | ---: | ---: | ---: |
| `resultsModel_ep1.csv` | 16 | 2.029 | 0.458 |
| `resultsModel_ep2.csv` | 32 | 3.228 | 0.447 |
| `resultsModel_ep3.csv` | 64 | 4.581 | 0.442 |


In [ ]:
plot_experiment_group("ep", "Número de épocas", historical=True)


### Inicialização de pesos

#### Registos históricos

Os valores abaixo são a última linha de cada CSV original. A coluna `val` foi calculada com o conjunto de teste reutilizado como validação; não representa um teste independente. Os CSV não guardam a configuração completa dos modelos.

| CSV | Épocas registadas | `val_loss` final | `val_acc` final |
| --- | ---: | ---: | ---: |
| `resultsModel_wis1.csv` | 16 | 1.795 | 0.360 |
| `resultsModel_wis2.csv` | 16 | 1.770 | 0.364 |
| `resultsModel_wis3.csv` | 16 | 1.751 | 0.378 |


In [ ]:
plot_experiment_group("wis", "Inicialização de pesos", historical=True)


### Taxa de dropout

#### Registos históricos

Os valores abaixo são a última linha de cada CSV original. A coluna `val` foi calculada com o conjunto de teste reutilizado como validação; não representa um teste independente. Os CSV não guardam a configuração completa dos modelos.

| CSV | Épocas registadas | `val_loss` final | `val_acc` final |
| --- | ---: | ---: | ---: |
| `resultsModel_dr1.csv` | 16 | 2.000 | 0.450 |
| `resultsModel_dr2.csv` | 16 | 1.559 | 0.457 |
| `resultsModel_dr3.csv` | 16 | 1.799 | 0.336 |


In [ ]:
plot_experiment_group("dr", "Taxa de dropout", historical=True)


### Pré-processamento


As imagens são convertidas para tons de cinzento, transformadas em tensores e normalizadas com `mean=0.5` e `std=0.5`. Este notebook não contém uma comparação experimental com e sem normalização.


## Conclusões

Os CSV históricos mostram diferenças entre as configurações registadas. No grupo da taxa de aprendizagem, `lr1` terminou com `val_acc` de 45,3%, enquanto `lr2` e `lr3` terminaram em 10,0%. Entre as funções de ativação, `af1` terminou em 45,1%, acima de `af2` (38,9%) e `af3` (39,9%). No grupo dos otimizadores, `op1` terminou em 45,6%, `op2` em 10,0% e `op3` em 39,1%. Estes valores descrevem as execuções guardadas, sem provar que uma escolha será sempre melhor.

Nos ensaios sobre número de épocas, `val_acc` passou de 45,8% (`ep1`, 16 épocas) para 44,2% (`ep3`, 64 épocas), enquanto `val_loss` subiu de 2,03 para 4,58. Os três batch sizes registaram `val_acc` final entre 34,0% e 35,4%, sem diferença clara nesses ficheiros. As inicializações ficaram entre 36,0% e 37,8%; no grupo de dropout, `dr2` terminou em 45,7%, próximo de `dr1` (45,0%) e acima de `dr3` (33,6%). O modelo de referência terminou em 46,2%, mas os grupos têm durações e configurações diferentes, pelo que este valor não estabelece uma classificação global.

Na execução recente, `default1` terminou em 44,6% de `val_acc`, enquanto `default2` e `default3` terminaram em 9,4%. Esta execução usa uma divisão correta entre treino e validação e **não é diretamente comparável** com os CSV históricos: nos ensaios antigos, o conjunto de teste foi usado como validação. Falta ainda avaliar os modelos recentes no conjunto de teste reservado.
